# Pre-process

In [ ]:
import os

# Paths for original jin-2025 dataset
PATHS = {'data_dir':  '/Users/catherinewang/Documents/School/genData/jin-2025/',
         'save_dir': '/Users/catherinewang/Documents/School/proximal-scCRISPR-github/jin-2025/saves',
         'util_dir': '/Users/catherinewang/Documents/School/proximal-scCRISPR-github/utils'}
PATHS['save_dir'] = PATHS['save_dir'] + '/3_L45glut'

In [ ]:
TAG = 'L45glut'

In [ ]:
if not 'jin2025_L45glut.h5ad' in os.listdir(PATHS['save_dir']):
    import numpy as np
    import pandas as pd
    import scanpy as sc
    

    # load and concatenate 2 datasets
    jin2025_10pert = sc.read_h5ad(PATHS['data_dir'] + 'subset_adata_SCARF_10celltypes_10perturbations.h5ad')
    jin2025_48pert = sc.read_h5ad(PATHS['data_dir'] + 'subset_adata_SCARF_10celltypes_48perturbations.h5ad')
    jin2025_10pert.obs_names = jin2025_10pert.obs_names + ':10pert' # change cellnames to specify data source
    jin2025_48pert.obs_names = jin2025_48pert.obs_names + ':48pert'

    jin2025 = sc.concat({'10pert': jin2025_10pert, '48pert': jin2025_48pert}, axis='obs') # combine into h5ad format

    # from prev analyses: choose the genes
    gene_dev_df = pd.read_csv(PATHS['save_dir'] + '/gene_dev.csv') # gene importance from scry deviance
    GENE_NUMNONZEROCELLS_MIN = 7500
    GENE_IMPORTANCERANK_MAX = 2000
    selected_genes = gene_dev_df.query(f'is_normalized_gene and \
                                        gene_nonzero_cells > {GENE_NUMNONZEROCELLS_MIN} \
                                        and (importance_rank <= {GENE_IMPORTANCERANK_MAX} or is_grna_target)')['gene_name'].to_list()

    # choose the glut cells
    glut_cells = jin2025.obs.query("neuron_type=='Glut'").index


    # subset for glut cells
    jin2025_glut = jin2025[glut_cells, selected_genes]

    # For the 399 NT grnas, reserve some for overall control cells and some for fake perturbations
    # For fake perturbations, combine 2 or 3 perturbations together
    # In the h5ad subset, create a new gene_target_w_fakeNT column that adds the new fake NT groupings
    # 
    NUM_SUBSAMPLE_CONTROL_CELLS = 10000
    NUM_NTPairs = 20    # will use up 2x(this number) grnas
    NUM_NTTriplets = 20 # will use up 3x(this number) grnas

    NT_names = jin2025_glut.obs.query("gene_target == 'Non_target'")['guide_call'].unique().to_numpy()
    
    
    np.random.seed(54321)
    # np.random.shuffle(NT_names)
    rng = np.random.default_rng()
    rng.shuffle(NT_names)


    fake_NT_dict = {} # dict of original guides to combined 

    # choose NT for pairs
    NT = NT_names
    for i in np.arange(NUM_NTPairs):
        fake_NT_name = NT[2*i]+'&'+NT[2*i+1]
        fake_NT_dict[NT[2*i]] = fake_NT_name
        fake_NT_dict[NT[2*i+1]] = fake_NT_name
    NT = NT[2*NUM_NTPairs:] # remove first 2*NUM_NTPairs

    # choose NT for triplets
    for i in np.arange(NUM_NTTriplets):
        fake_NT_name = NT[3*i]+'&'+NT[3*i+1]+'&'+NT[3*i+2]
        fake_NT_dict[NT[3*i]] = fake_NT_name
        fake_NT_dict[NT[3*i+1]] = fake_NT_name
        fake_NT_dict[NT[3*i+2]] = fake_NT_name
    NT = NT[3*NUM_NTTriplets:] # remove first 3*NUM_NTTriplets

    NT_controls = NT

    # pool Non_target for the rest
    for nt_ in NT:
        fake_NT_dict[nt_] = 'Non_target'

    del NT # no more remaining

    gene_target_w_fakeNT = [jin2025_glut.obs['gene_target'].iloc[i, ] if (fake_NT_dict.get(jin2025_glut.obs['guide_call'].iloc[i, ])==None) 
                            else fake_NT_dict.get(jin2025_glut.obs['guide_call'].iloc[i, ]) 
                            for i in np.arange(jin2025_glut.obs.shape[0])]

    jin2025_glut.obs['gene_target_w_fakeNT'] = gene_target_w_fakeNT


    # subsample controls
    trt_cells   = jin2025_glut.obs.query("gene_target_w_fakeNT != 'Non_target'").index.tolist()
    cntrl_cells = jin2025_glut.obs[jin2025_glut.obs['guide_call'].isin(NT_controls)].index
    cntrl_cells = rng.choice(cntrl_cells, size=NUM_SUBSAMPLE_CONTROL_CELLS, replace=False).tolist()
    subset_cells = trt_cells + cntrl_cells

    # subset
    jin2025_subset = jin2025_glut[subset_cells, selected_genes]

    # save
    jin2025_subset.write_h5ad(
        PATHS['save_dir'] + '/jin2025_glut.h5ad',
        compression='lzf'
    )




In [3]:
# # For the 399 NT grnas, reserve some for overall control cells and some for fake perturbations
# # For fake perturbations, combine 2 or 3 perturbations together
# # In the h5ad subset, create a new gene_target_w_fakeNT column that adds the new fake NT groupings
# # 
# NUM_SUBSAMPLE_CONTROL_CELLS = 10000
# NUM_NTPairs = 20    # will use up 2x(this number) grnas
# NUM_NTTriplets = 20 # will use up 3x(this number) grnas

# np.random.seed(54321)
# rng = np.random.default_rng()
# NT_names = jin2025_glut.obs.query("gene_target == 'Non_target'")['guide_call'].unique().to_numpy()
# # np.random.shuffle(NT_names)
# rng.shuffle(NT_names)


# fake_NT_dict = {} # dict of original guides to combined 

# # choose NT for pairs
# NT = NT_names
# for i in np.arange(NUM_NTPairs):
#     fake_NT_name = NT[2*i]+'&'+NT[2*i+1]
#     fake_NT_dict[NT[2*i]] = fake_NT_name
#     fake_NT_dict[NT[2*i+1]] = fake_NT_name
# NT = NT[2*NUM_NTPairs:] # remove first 2*NUM_NTPairs

# # choose NT for triplets
# for i in np.arange(NUM_NTTriplets):
#     fake_NT_name = NT[3*i]+'&'+NT[3*i+1]+'&'+NT[3*i+2]
#     fake_NT_dict[NT[3*i]] = fake_NT_name
#     fake_NT_dict[NT[3*i+1]] = fake_NT_name
#     fake_NT_dict[NT[3*i+2]] = fake_NT_name
# NT = NT[3*NUM_NTTriplets:] # remove first 3*NUM_NTTriplets

# NT_controls = NT

# # pool Non_target for the rest
# for nt_ in NT:
#     fake_NT_dict[nt_] = 'Non_target'

# del NT # no more remaining

# gene_target_w_fakeNT = [jin2025_glut.obs['gene_target'].iloc[i, ] if (fake_NT_dict.get(jin2025_glut.obs['guide_call'].iloc[i, ])==None) 
#                         else fake_NT_dict.get(jin2025_glut.obs['guide_call'].iloc[i, ]) 
#                         for i in np.arange(jin2025_glut.obs.shape[0])]

# jin2025_glut.obs['gene_target_w_fakeNT'] = gene_target_w_fakeNT


# # subsample controls
# trt_cells   = jin2025_glut.obs.query("gene_target_w_fakeNT != 'Non_target'").index.tolist()
# cntrl_cells = jin2025_glut.obs[jin2025_glut.obs['guide_call'].isin(NT_controls)].index
# cntrl_cells = rng.choice(cntrl_cells, size=NUM_SUBSAMPLE_CONTROL_CELLS, replace=False).tolist()
# subset_cells = trt_cells + cntrl_cells

# # subset
# jin2025_subset = jin2025_glut[subset_cells, selected_genes]

# # save
# jin2025_subset.write_h5ad(
#     PATHS['save_dir'] + '/jin2025_glut.h5ad',
#     compression='lzf'
# )

In [4]:
# jin2025_subset.obs.loc[:, ['guide_call', 'gene_target', 'gene_target_w_fakeNT']]

In [5]:
# jin2025_subset.obs.groupby('gene_target_w_fakeNT').count().iloc[:, 1].sort_values()

Concatenate the two datasets

In [6]:
# jin2025_10pert = sc.read_h5ad(PATHS['data_dir'] + 'subset_adata_SCARF_10celltypes_10perturbations.h5ad')
# jin2025_48pert = sc.read_h5ad(PATHS['data_dir'] + 'subset_adata_SCARF_10celltypes_48perturbations.h5ad')
# jin2025_10pert.obs_names = jin2025_10pert.obs_names + ':10pert' # change cellnames to specify data source
# jin2025_48pert.obs_names = jin2025_48pert.obs_names + ':48pert'

# jin2025 = sc.concat({'10pert': jin2025_10pert, '48pert': jin2025_48pert}, axis='obs') # combine into h5ad format

Subselect cells (glut cells) and genes (high importance, at least x cells expressed etc...)

In [7]:
# # from prev analyses: choose the genes
# gene_dev_df = pd.read_csv(PATHS['save_dir'] + '/gene_dev.csv') # gene importance from scry deviance
# GENE_NUMNONZEROCELLS_MIN = 7500
# GENE_IMPORTANCERANK_MAX = 2000
# selected_genes = gene_dev_df.query(f'is_normalized_gene and \
#                                      gene_nonzero_cells > {GENE_NUMNONZEROCELLS_MIN} \
#                                      and (importance_rank <= {GENE_IMPORTANCERANK_MAX} or is_grna_target)')['gene_name'].to_list()

# # choose the cells (glut cells)
# glut_cells = jin2025.obs.query("neuron_type=='Glut'").index

# # subset
# jin2025_subset = jin2025[glut_cells, selected_genes]

# # save
# jin2025_subset = jin2025[glut_cells, selected_genes]
# jin2025_subset.write_h5ad(
#     PATHS['save_dir'] + '/jin2025_glut.h5ad',
#     compression='lzf'
# )

In [8]:
# oops I analyzed all glut cells, not just the subtype 
# jin2025.obs.head()
# jin2025.obs.columns
# jin2025.obs.loc[:, ['predicted_class', 'predicted_subclass', 'predicted_supertype', 'predicted_cluster', 'predicted_group']]
# jin2025.obs.query("predicted_group=='005 L4-5 IT CTX Glut'") # oops I tested on all Glut cells, not just this subtype 

In [9]:
# from pathlib import Path
# # Path(__file__).parent.resolve()
# # relative paths to replace in pipelin.py
# (Path('.')/'../../../../../genData/jin-2025/').resolve()
# (Path('.')/'../../../saves/').resolve()
# # absolute paths
# PATHS = {'data_dir':  '/Users/catherinewang/Documents/School/genData/jin-2025/',
#          'save_dir': '/Users/catherinewang/Documents/School/proximal-scCRISPR-github/jin-2025/saves',
#          'util_dir': '/Users/catherinewang/Documents/School/proximal-scCRISPR-github/utils'}

# Subsample
subsample control cells and create fake NT grnas

In [10]:
# import numpy as np
# import pandas as pd
# import scanpy as sc


# # load and concatenate 2 datasets
# jin2025_10pert = sc.read_h5ad(PATHS['data_dir'] + 'subset_adata_SCARF_10celltypes_10perturbations.h5ad')
# jin2025_48pert = sc.read_h5ad(PATHS['data_dir'] + 'subset_adata_SCARF_10celltypes_48perturbations.h5ad')
# jin2025_10pert.obs_names = jin2025_10pert.obs_names + ':10pert' # change cellnames to specify data source
# jin2025_48pert.obs_names = jin2025_48pert.obs_names + ':48pert'

# jin2025 = sc.concat({'10pert': jin2025_10pert, '48pert': jin2025_48pert}, axis='obs') # combine into h5ad format

In [11]:
# # For the 399 NT grnas, reserve some for overall control cells and some for fake perturbations
# # For fake perturbations, combine 2 or 3 perturbations together
# # In the h5ad subset, create a new gene_target_w_fakeNT column that adds the new fake NT groupings
# # 
# NUM_SUBSAMPLE_CONTROL_CELLS = 10000
# NUM_NTPairs = 20    # will use up 2x(this number) grnas
# NUM_NTTriplets = 20 # will use up 3x(this number) grnas

# np.random.seed(54321)
# NT_names = jin2025.obs.query("gene_target == 'Non_target'")['guide_call'].unique().to_numpy()
# np.random.shuffle(NT_names)



# NT = NT_names
# NTPairs    = [NT[2*i]+'&'+NT[2*i+1] for i in np.arange(NUM_NTPairs)]
# NT = NT[2*NUM_NTPairs:] # remove first 2*NUM_NTPairs
# NTTriplets = [NT[3*i]+'&'+NT[3*i+1]+'&'+NT[3*i+2] for i in np.arange(NUM_NTTriplets)]
# NT = NT[3*NUM_NTTriplets:] # remove first 3*NUM_NTTriplets
# NT_controls = NT

# # (NUM_NTPairs * 2 + NUM_NTTriplets * 3)
# # np.perm(NT_names)



# # NT_grnas_for_control = sample(NT_names, size = round(.75*length(NT_names)) )
# # NT_grnas_for_null    = setdiff(NT_names, NT_grnas_for_control)

In [12]:
# fake_NT_dict = {} # dict of original guides to combined 

# # choose NT for pairs
# NT = NT_names
# for i in np.arange(NUM_NTPairs):
#     fake_NT_name = NT[2*i]+'&'+NT[2*i+1]
#     fake_NT_dict[NT[2*i]] = fake_NT_name
#     fake_NT_dict[NT[2*i+1]] = fake_NT_name
# NT = NT[2*NUM_NTPairs:] # remove first 2*NUM_NTPairs

# # choose NT for triplets
# for i in np.arange(NUM_NTTriplets):
#     fake_NT_name = NT[3*i]+'&'+NT[3*i+1]+'&'+NT[3*i+2]
#     fake_NT_dict[NT[3*i]] = fake_NT_name
#     fake_NT_dict[NT[3*i+1]] = fake_NT_name
#     fake_NT_dict[NT[3*i+2]] = fake_NT_name
# NT = NT[3*NUM_NTTriplets:] # remove first 3*NUM_NTTriplets

# # pool Non_target for the rest
# for nt_ in NT:
#     fake_NT_dict[nt_] = 'Non_target'

# del NT # no more remaining


# # gene_target_w_fakeNT = jin2025.obs['gene_target'].copy()
# gene_target_w_fakeNT = [x if (fake_NT_dict.get(x)==None) else fake_NT_dict.get(x) for x in jin2025.obs['guide_call']]

# jin2025.obs['gene_target_w_fakeNT'] = gene_target_w_fakeNT